# TRANSFORM (T) STEP
- Match titles accounting for minor diffrences. Ensure all Transcripts are paired with their coresponding talk. 

In [ ]:
# --- 2. TRANSFORMATION (T) ---
print("\n--- Step 2: Transformation (T) ---")

def closest_transcript_title(title):
    """Fuzzy-matches transcript titles against schedule titles. 
    Returns: Best match, its score, and index in the list of transcript titles.
    Used """
    match = process.extractOne(
        title,
        transcript_titles,
        scorer=fuzz.WRatio,
        score_cutoff=90
    )
    if match:
        return match[0], match[1], match[2]
    return None, 0, None

def closest_schedule_title(title):
    """Find closest schedule title for unmatched transcripts
    Returns: Closest schedule title, its type, and similarity score."""
    match = process.extractOne(
        title,
        schedule_df["title"].dropna().tolist(),
        scorer=fuzz.WRatio
    )
    if match:
        row = schedule_df[schedule_df["title"] == match[0]].iloc[0]
        return match[0], row["type"], match[1]
    return None, None, 0

# Prepare transcript titles for matching
transcript_titles = transcripts_df["title"].dropna().tolist()


# --- DATA PREPERATION ---
# Separate talks and performances from other schedule entries. Testing and context indicates these are the only categories for transcripts
matchable_events = schedule_df[schedule_df["type"].isin(["talk", "performance"])].copy()
other_events = schedule_df[~schedule_df["type"].isin(["talk", "performance"])].copy()
print(f"Talks and performances: {len(matchable_events)}")

# Collect list of transcript titles for later troubleshooting
transcript_titles = transcripts_df["title"].dropna().tolist()


# --- MATCH TITLES ---
# Fuzzy-match schedule titles against transcript titles
matchable_events[["matched_title", "match_score", "match_index"]] = (
    matchable_events["title"].apply(
        lambda x: pd.Series(closest_transcript_title(x))
    )
)

# --- ENRICH DATA ---
# Merge transcripts onto talks and performances
enriched_events = pd.merge(
    matchable_events,
    transcripts_df,
    left_on="matched_title",
    right_on="title",
    how="left",
    suffixes=("_schedule", "_transcript")
)


# --- IDENTIFY UNMATCHED TRANSCRIPTS ---
# Capture matched transcript titles BEFORE cleaning up the merged DataFrame
matched_titles = enriched_events["title_transcript"].dropna().unique()
# Find genuinely unmatched transcripts
unmatched_transcripts = transcripts_df[~transcripts_df["title"].isin(matched_titles)].copy()

# Find closest schedule title and event type for unmatched transcripts
results = unmatched_transcripts["title"].apply(closest_schedule_title).tolist()


# --- MATCHING STATS AND TROUBLESHOOTING ---
results_df = pd.DataFrame(
    results,
    columns=[
        "closest_schedule_title",
        "schedule_type",
        "similarity_score"
    ],
    index=unmatched_transcripts.index
)
unmatched_transcripts[["closest_schedule_title", "schedule_type", "similarity_score"]] = results_df

# Display unmatched transcripts for troubleshooting
print("\nDataframe of Transcripts where a coresponding event in schedule could not be found")
display(
    unmatched_transcripts[
        [
            "title",
            "closest_schedule_title",
            "schedule_type",
            "similarity_score"
        ]
    ].sort_values(
        "similarity_score",
        ascending=False
    )
)
matched_count = enriched_events["transcript"].notna().sum()
unmatched_count = len(unmatched_transcripts)
total_transcripts = len(transcripts_df)

print(f"Total transcripts: {total_transcripts}")
print(f"Matched transcripts: {matched_count}")
print(f"Unmatched transcripts: {unmatched_count}")
print(f"Match rate: {matched_count / total_transcripts * 100:.1f}%")


# --- CLEANUP AND RECOMBINE ---
# Remove temporary matching columns and duplicate transcript title
enriched_events = enriched_events.drop(
    columns=[
        "title_transcript",
        "matched_title",
        "match_score",
        "match_index"
    ],
    errors="ignore"
)
# Rename original schedule title back to title
enriched_events = enriched_events.rename(
    columns={"title_schedule": "title"}
)
# Recombine enriched talks and performances with other schedule entries
enriched_schedule = pd.concat(
    [enriched_events, other_events],
    ignore_index=True,
    sort=False
)

display(enriched_schedule.iloc[1])

# Transform (Part 2): Expand Combined Schedule/Talk Table to unique Events.
- (Work in progress)
- occurences column contains JSON which includes details about each instance where the event ran. 
    - In cases where event ran multiple times this coresponds to multiple date times and other data. 
    - The goal of this section is to expand the new enriched Scedule table to have one occurence in time per row with each row having its own start and finish time. 
    - Table should also derive a flag for multiple occurences to help with analysis down the line preventing doublecounting. 

In [ ]:
# T2: IDENTIFY MULTIPLE OCCURRENCES AND EXPAND SCHEDULE

# ---------------------------------------------------------
# 1. IMPORT, VALIDATE AND IDENTIFY MULTIPLE OCCURRENCES
# ---------------------------------------------------------
enriched_schedule["occurrences"] = enriched_schedule["occurrences"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
assert enriched_schedule["occurrences"].apply(
    lambda x: isinstance(x, list) and all(isinstance(i, dict) for i in x)
).all(), "Invalid occurrences structure."

enriched_schedule["multiple_occurrences"] = enriched_schedule[
    "occurrences"   
].str.len().gt(1).map({True: "Y", False: "N"})


# ---------------------------------------------------------
# 2. EXPAND SCHEDULE INTO INDIVIDUAL OCCURRENCE ROWS
# ---------------------------------------------------------
expanded_rows = [ # This pattern is nested list comprehension, It is a dense way to get all of the itterable parts in one line. It is equivalent to a nested for loop but more compact.
    {
        **row.drop( ## Drop the columns that will be replaced by occurrence-specific data
            ["occurrences", "start_date", "end_date", "occurrence_data"],
            errors="ignore" # Catches case where these columns don't exist in the DataFrame yet
        ).to_dict(),

        "source_row_index": index,
        "occurrence_data": occurrence.copy() if occurrence else None,
        "occurrence_number": number if occurrence else pd.NA,
        "occurrence_count": len(occurrences),
        "start_date": occurrence.get("start_date") if occurrence else pd.NaT,
        "end_date": occurrence.get("end_date") if occurrence else pd.NaT
    }

    for index, row in enriched_schedule.iterrows()
    for occurrences in [row["occurrences"]]
    for number, occurrence in enumerate(occurrences or [None], start=1)
]

expanded_schedule = pd.DataFrame(expanded_rows)

expanded_schedule[["start_date", "end_date"]] = expanded_schedule[
    ["start_date", "end_date"]
].apply(pd.to_datetime, errors="raise")


# ---------------------------------------------------------
# 3. CREATE MULTIPLE OCCURRENCES TROUBLESHOOTING DATAFRAME
# ---------------------------------------------------------
multiple_occurrences_df = expanded_schedule.query(
    "multiple_occurrences == 'Y'"
).copy()

# ---------------------------------------------------------
# 4. TRANSFORMATION RESULTS AND VALIDATION
# ---------------------------------------------------------
print("\n--- T2 OCCURRENCE EXPANSION RESULTS ---")

print(f"Original rows: {len(enriched_schedule)}")
print(f"Expanded rows: {len(expanded_schedule)}")
print(f"Additional rows: {len(expanded_schedule) - len(enriched_schedule)}")
print(f"Size of multiple occurrences DataFrame: {len(multiple_occurrences_df)}")

print("\n--- DATE COMPLETENESS ---")
for column in ["start_date", "end_date"]:
    print(f"{column}: {expanded_schedule[column].notna().sum()} populated | {expanded_schedule[column].isna().sum()} missing")
assert len(expanded_schedule) == sum(
    max(1, len(x)) for x in enriched_schedule["occurrences"]
)

assert pd.api.types.is_datetime64_any_dtype(expanded_schedule["start_date"])
assert pd.api.types.is_datetime64_any_dtype(expanded_schedule["end_date"])

assert (multiple_occurrences_df["multiple_occurrences"] == "Y").all()

print("\nT2 TRANSFORMATION COMPLETE.")


# ---------------------------------------------------------
# 5. DISPLAY RESULTS AND EXPORT DTYPES
# ---------------------------------------------------------
Abridges_columns = ["id", "type","names","title","short_description","video_privacy","transcript",
    "multiple_occurrences","start_date","end_date","occurrence_data","occurrence_number",
    "occurrence_count"]

print("\n Display head of new expanded schedule.")
display(expanded_schedule[Abridges_columns].head(5))

print("\n Display rows with multiple occurrences for troubleshooting.")
display(multiple_occurrences_df[Abridges_columns].head(5))

print("\n showing dtypes of expanded schedule DataFrame:")
dtype_report = "\n".join(
    f"{i}. {col}: {dtype}"
    for i, (col, dtype) in enumerate(expanded_schedule.dtypes.items(), 1)
)

print("\n Missing date Troublehsooting")
display(
    expanded_schedule[expanded_schedule["start_date"].isna() | expanded_schedule["end_date"].isna()][Abridges_columns]
)

print(dtype_report)

In [ ]:
# T3: Additional Derived Columns
expanded_schedule["Duration_min"] = expanded_schedule.apply(
    lambda row: (row["end_date"] - row["start_date"]).total_seconds() / 60
    if pd.notna(row["start_date"]) and pd.notna(row["end_date"])
    else pd.NA,
    axis=1
)
display(expanded_schedule[["start_date", "end_date", "Duration_min"]].head(5))

# Confirm column datatypes
print("\n showing dtypes of expanded schedule DataFrame:")
dtype_report = "\n".join(
    f"{i}. {col}: {dtype}"
    for i, (col, dtype) in enumerate(expanded_schedule.dtypes.items(), 1)
)
print(dtype_report)